# One-Stage Daily Raw Rich Feature Engineering

这份 notebook 只做 richer `single-stage` 所需的 daily 特征工程，不进入建模。

目标：
- 从 `event_stage1_aligned_features_v1.csv` 读取事件级原料
- 按 `target_trade_date` 聚合 richer daily raw features
- 与当前 `Stage 2` 相同的 daily base / split / target 对齐
- 产出后续 richer single-stage ablation 可直接使用的 daily 表

当前 notebook 不做：
- nested CV
- 模型比较
- 调参
- SHAP
- two-stage 结果刷新

## Output Design

这版会导出 3 份主要结果：

- `one_stage_daily_raw_rich_event_daily_bridge_v1.csv`
- `one_stage_daily_raw_rich_features_v1.csv`
- `one_stage_daily_raw_rich_feature_qc_summary_v1.csv`

说明：
- richer event-level 原料来自 `data/integration/event_stage1_aligned_features_v1.csv`
- daily base 来自 `stage2/artifacts/stage2_daily_features_wide_v2.csv`
- 这样 richer single-stage 与当前 two-stage 主版本仍然共用同一组：
  - `date`
  - `stage2_split`
  - `gold_ret_1d`
  - core lagged controls
  - trend lagged features

# Main Rerun One-Stage Raw-Rich Feature Runtime Config

这个 notebook 固定用于本次 `main_rerun` 的 richer single-stage feature engineering：

- `STAGE_SPLIT_SCHEME = "main_rerun_primary_holdout"`
- 正式 `primary holdout = 2025-07-15 ~ 2025-10-25`
- daily base 固定读取 `main_rerun` 的 Stage 2 wide 表
- 输出目录固定到 `main_rerun/artifacts/ablation/one_stage_daily_raw_rich/`
- 不覆盖旧 `ablation/artifacts/`

In [1]:
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "plan_v2.md").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)

MAIN_RERUN_STAGE2_DIR = PROJECT_ROOT / "main_rerun" / "artifacts" / "stage2"
MAIN_RERUN_ABLATION_DIR = PROJECT_ROOT / "main_rerun" / "artifacts" / "ablation" / "one_stage_daily_raw_rich"
MAIN_RERUN_ABLATION_DIR.mkdir(parents=True, exist_ok=True)

os.environ["STAGE_SPLIT_SCHEME"] = "main_rerun_primary_holdout"
os.environ["STAGE2_HOLDOUT_LABEL"] = "primary_holdout"
os.environ["STAGE2_BASE_PATH"] = str(
    MAIN_RERUN_STAGE2_DIR / "stage2_daily_features_wide_v2_main_rerun.csv"
)
os.environ["ONE_STAGE_ARTIFACTS_DIR"] = str(MAIN_RERUN_ABLATION_DIR)

print("Project root:", PROJECT_ROOT)
print("Main rerun Stage 2 base dir:", MAIN_RERUN_STAGE2_DIR)
print("Main rerun ablation output dir:", MAIN_RERUN_ABLATION_DIR)

Project root: /Users/horace/Coding/ML finance/project
Main rerun Stage 2 base dir: /Users/horace/Coding/ML finance/project/main_rerun/artifacts/stage2
Main rerun ablation output dir: /Users/horace/Coding/ML finance/project/main_rerun/artifacts/ablation/one_stage_daily_raw_rich


In [2]:
from pathlib import Path
import os

os.environ["MPLCONFIGDIR"] = str(Path.cwd() / ".mpl-cache")

import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="talk")

ROOT = Path.cwd()
if ROOT.name == "ablation":
    ROOT = ROOT.parent

ARTIFACTS_DIR = Path(os.environ.get("ONE_STAGE_ARTIFACTS_DIR", ROOT / "ablation" / "artifacts"))
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

EVENT_PATH = ROOT / "data" / "integration" / "event_stage1_aligned_features_v1.csv"
STAGE2_BASE_PATH = Path(
    os.environ.get(
        "STAGE2_BASE_PATH",
        ROOT / "stage2" / "artifacts" / "stage2_daily_features_wide_v2.csv",
    )
)

BRIDGE_OUTPUT_PATH = ARTIFACTS_DIR / "one_stage_daily_raw_rich_event_daily_bridge_v1.csv"
DAILY_OUTPUT_PATH = ARTIFACTS_DIR / "one_stage_daily_raw_rich_features_v1.csv"
QC_OUTPUT_PATH = ARTIFACTS_DIR / "one_stage_daily_raw_rich_feature_qc_summary_v1.csv"

NEG_THRESHOLD = 0.50
POS_THRESHOLD = 0.50

CORE_CONTROL_FEATURES = [
    "dxy_ret_1d_lag1",
    "real_yield_change_5d_lag1",
    "vix_ma_5d_lag1",
    "vix_change_5d_lag1",
    "spx_ret_1d_lag1",
    "spx_drawdown_5d_lag1",
    "gold_spx_corr_20d_lag1",
]

TREND_FEATURES = [
    "trend_tariff_z_60d_lag1",
    "trend_inflation_z_60d_lag1",
    "trend_war_z_60d_lag1",
    "trend_tariff_spike_lag1",
    "trend_inflation_spike_lag1",
    "trend_war_spike_lag1",
]

FORBIDDEN_FEATURE_COLS = [
    "label_retreat",
    "p_taco",
    "follow_up_count_all_48h",
    "follow_up_count_relevant_48h",
    "theme_hits",
    "action_hits",
    "object_hits",
]

In [3]:
SPLIT_SCHEME = os.environ.get("STAGE_SPLIT_SCHEME", "primary_holdout")

TRAIN_LABEL = "train_pool"
HOLDOUT_LABEL = os.environ.get("STAGE2_HOLDOUT_LABEL", "primary_holdout")
OUTSIDE_LABEL = "outside_main_sample"

SPLIT_SCHEMES = {
    "primary_holdout": {
        "train_ranges": [
            ("2017-01-20", "2021-01-08"),
            ("2025-01-20", "2025-07-31"),
        ],
        "holdout_range": ("2025-08-01", "2025-10-25"),
    },
    "main_rerun_primary_holdout": {
        "train_ranges": [
            ("2017-01-20", "2021-01-08"),
            ("2025-01-20", "2025-07-14"),
        ],
        "holdout_range": ("2025-07-15", "2025-10-25"),
    },
    "first_term_final_year_holdout": {
        "train_ranges": [
            ("2017-01-20", "2019-06-30"),
        ],
        "holdout_range": ("2019-07-01", "2020-02-28"),
    },
}


def describe_split_scheme(split_scheme):
    scheme = SPLIT_SCHEMES[split_scheme]
    return {
        "split_scheme": split_scheme,
        "train_ranges": scheme["train_ranges"],
        "holdout_range": scheme["holdout_range"],
        "train_label": TRAIN_LABEL,
        "holdout_label": HOLDOUT_LABEL,
    }


def classify_daily_split(date_values, split_scheme=SPLIT_SCHEME):
    scheme = SPLIT_SCHEMES[split_scheme]
    dates = pd.to_datetime(pd.Series(date_values)).dt.normalize()
    tz = getattr(dates.dt, "tz", None)
    if tz is not None:
        dates = dates.dt.tz_convert(None)

    train_mask = pd.Series(False, index=dates.index)
    for start_text, end_text in scheme["train_ranges"]:
        start = pd.Timestamp(start_text)
        end = pd.Timestamp(end_text)
        train_mask |= (dates >= start) & (dates <= end)

    holdout_start, holdout_end = scheme["holdout_range"]
    holdout_mask = (dates >= pd.Timestamp(holdout_start)) & (dates <= pd.Timestamp(holdout_end))
    return np.select(
        [train_mask, holdout_mask],
        [TRAIN_LABEL, HOLDOUT_LABEL],
        default=OUTSIDE_LABEL,
    )


print("Split scheme:", describe_split_scheme(SPLIT_SCHEME))

Split scheme: {'split_scheme': 'main_rerun_primary_holdout', 'train_ranges': [('2017-01-20', '2021-01-08'), ('2025-01-20', '2025-07-14')], 'holdout_range': ('2025-07-15', '2025-10-25'), 'train_label': 'train_pool', 'holdout_label': 'primary_holdout'}


## Load And Normalize Inputs

这一步只做：

- 读取 event-level 原料
- 统一时间解析
- 规范布尔列
- 读取 daily base
- 检查 `stage2_split` 是否与 protocol 一致

In [4]:
def classify_stage2_date(date_series):
    return classify_daily_split(
        date_series,
        split_scheme=SPLIT_SCHEME,
    )


def load_event_source():
    df = pd.read_csv(EVENT_PATH)

    df["event_time_utc"] = pd.to_datetime(df["event_time_utc"], format="mixed", utc=True)
    df["feature_anchor_date"] = pd.to_datetime(df["feature_anchor_date"], format="mixed", utc=True)
    df["target_trade_date"] = pd.to_datetime(df["target_trade_date"], format="mixed", utc=True)
    df["target_trade_date_naive"] = df["target_trade_date"].dt.tz_convert(None).dt.normalize()
    df["event_date_ny"] = df["event_time_utc"].dt.tz_convert("America/New_York").dt.normalize()

    bool_defaults = {
        "is_trump_direct_text": False,
        "is_retweet": False,
        "in_market_hours": False,
    }
    for col, default in bool_defaults.items():
        if col in df.columns:
            df[col] = df[col].fillna(default).astype(bool)
        else:
            df[col] = default

    numeric_defaults = {
        "keyword_score": 0.0,
        "finbert_pos": 0.0,
        "finbert_neu": 0.0,
        "finbert_neg": 0.0,
    }
    for col, default in numeric_defaults.items():
        if col not in df.columns:
            raise ValueError(f"Missing required event column: {col}")
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(default)

    return df


def load_stage2_daily_base():
    df = pd.read_csv(STAGE2_BASE_PATH)
    df["date"] = pd.to_datetime(df["date"]).dt.normalize()

    expected_split = classify_stage2_date(df["date"])
    mismatch_count = int((df["stage2_split"] != expected_split).sum())
    print("stage2 split mismatches vs protocol:", mismatch_count)

    keep_cols = [
        "date",
        "stage2_split",
        "is_main_stage2_sample",
        "gold_close",
        "gold_ret_1d",
    ] + CORE_CONTROL_FEATURES + TREND_FEATURES

    missing_keep_cols = [col for col in keep_cols if col not in df.columns]
    if missing_keep_cols:
        raise ValueError(f"Missing stage2 base columns: {missing_keep_cols}")

    base = (
        df[keep_cols]
        .drop_duplicates(subset=["date"])
        .sort_values("date")
        .reset_index(drop=True)
    )
    return base, mismatch_count

In [5]:
event_df = load_event_source()
daily_base, split_mismatch_count = load_stage2_daily_base()

print("event_df rows:", event_df.shape)
print("daily_base rows:", daily_base.shape)
display(event_df[[
    "event_id",
    "event_time_utc",
    "target_trade_date_naive",
    "finbert_pos",
    "finbert_neu",
    "finbert_neg",
    "candidate_priority" if "candidate_priority" in event_df.columns else "source",
]].head())
display(daily_base.head())

stage2 split mismatches vs protocol: 0
event_df rows: (1886, 86)
daily_base rows: (2293, 18)


,event_id,event_time_utc,target_trade_date_naive,finbert_pos,finbert_neu,finbert_neg,candidate_priority
0,FIRST_TERM_TWITTER_824440456813707265,2017-01-26 02:14:56+00:00,2017-01-26,0.037903,0.917480,0.044617,low
1,FIRST_TERM_TWITTER_824615820391305216,2017-01-26 13:51:46+00:00,2017-01-26,0.048340,0.094849,0.856934,low
2,FIRST_TERM_TWITTER_824616644370714627,2017-01-26 13:55:03+00:00,2017-01-26,0.007835,0.033997,0.958008,high
3,FIRST_TERM_TWITTER_824970003153842176,2017-01-27 13:19:10+00:00,2017-01-27,0.063843,0.144287,0.791992,high
4,FIRST_TERM_TWITTER_825017279209410561,2017-01-27 16:27:02+00:00,2017-01-27,0.314453,0.672363,0.013039,low


,date,stage2_split,is_main_stage2_sample,gold_close,gold_ret_1d,dxy_ret_1d_lag1,real_yield_change_5d_lag1,vix_ma_5d_lag1,vix_change_5d_lag1,spx_ret_1d_lag1,spx_drawdown_5d_lag1,gold_spx_corr_20d_lag1,trend_tariff_z_60d_lag1,trend_inflation_z_60d_lag1,trend_war_z_60d_lag1,trend_tariff_spike_lag1,trend_inflation_spike_lag1,trend_war_spike_lag1
0,2016-11-01,outside_main_sample,0,1302.519715,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2016-11-02,outside_main_sample,0,1308.589123,0.004649,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0
2,2016-11-03,outside_main_sample,0,1311.193116,0.001988,-0.000622,NaN,NaN,NaN,-0.006547,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0
3,2016-11-04,outside_main_sample,0,1311.596841,0.000308,-0.001354,NaN,19.986667,NaN,-0.004433,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0
4,2016-11-07,outside_main_sample,0,1299.637957,-0.009160,-0.001615,NaN,20.617500,NaN,-0.001668,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0


## Build Event-Level Helper Columns

这一步只构造 richer raw aggregation 所需的 event-level helper columns，
不直接产出 modeling table。

In [6]:
def add_event_level_helper_columns(df):
    out = df.copy()

    if "candidate_priority" in out.columns:
        priority = out["candidate_priority"].astype(str).str.strip().str.lower()
        out["candidate_priority_high_flag"] = priority.eq("high").astype(int)
    else:
        out["candidate_priority_high_flag"] = 0

    out["keyword_score_filled"] = pd.to_numeric(out.get("keyword_score", 0.0), errors="coerce").fillna(0.0)

    source_norm = out.get("source", pd.Series("", index=out.index)).astype(str).str.strip().str.lower()
    out["source_twitter_flag"] = source_norm.eq("twitter").astype(int)
    out["source_truthsocial_flag"] = source_norm.eq("truthsocial").astype(int)

    term_norm = out.get("term_id", pd.Series("", index=out.index)).astype(str).str.strip().str.lower()
    out["term_first_flag"] = term_norm.eq("first_term").astype(int)
    out["term_second_flag"] = term_norm.eq("second_term").astype(int)

    out["finbert_pos"] = out["finbert_pos"].clip(0.0, 1.0)
    out["finbert_neu"] = out["finbert_neu"].clip(0.0, 1.0)
    out["finbert_neg"] = out["finbert_neg"].clip(0.0, 1.0)

    out["extreme_neg_flag"] = (out["finbert_neg"] >= NEG_THRESHOLD).astype(int)
    out["extreme_pos_flag"] = (out["finbert_pos"] >= POS_THRESHOLD).astype(int)

    return out


event_df_enriched = add_event_level_helper_columns(event_df)

helper_cols = [
    "candidate_priority_high_flag",
    "keyword_score_filled",
    "source_twitter_flag",
    "source_truthsocial_flag",
    "term_first_flag",
    "term_second_flag",
    "extreme_neg_flag",
    "extreme_pos_flag",
]
display(event_df_enriched[["event_id"] + helper_cols].head())

,event_id,candidate_priority_high_flag,keyword_score_filled,source_twitter_flag,source_truthsocial_flag,term_first_flag,term_second_flag,extreme_neg_flag,extreme_pos_flag
0,FIRST_TERM_TWITTER_824440456813707265,0,2,1,0,1,0,0,0
1,FIRST_TERM_TWITTER_824615820391305216,0,6,1,0,1,0,1,0
2,FIRST_TERM_TWITTER_824616644370714627,1,3,1,0,1,0,1,0
3,FIRST_TERM_TWITTER_824970003153842176,1,6,1,0,1,0,1,0
4,FIRST_TERM_TWITTER_825017279209410561,0,2,1,0,1,0,0,0


## Aggregate Rich Daily Raw Features

richer single-stage 的核心是：

- 保留 `target_trade_date` 级别聚合
- 但不再只做窄口径的 negative sentiment
- 而是把 richer FinBERT / event structure / priority / source / term 组都带下来

In [7]:
def aggregate_daily_raw_rich_features(events):
    grouped_rows = []

    for target_date, group in events.groupby("target_trade_date_naive"):
        pos = group["finbert_pos"].astype(float)
        neu = group["finbert_neu"].astype(float)
        neg = group["finbert_neg"].astype(float)

        grouped_rows.append(
            {
                "date": target_date,
                "finbert_pos_max": float(pos.max()),
                "finbert_pos_mean": float(pos.mean()),
                "finbert_neg_max": float(neg.max()),
                "finbert_neg_mean": float(neg.mean()),
                "finbert_neu_mean": float(neu.mean()),
                "finbert_neg_top1": float(neg.max()),
                "finbert_neg_top2_sum": float(neg.nlargest(min(2, len(neg))).sum()),
                "finbert_neg_any": float(1.0 - np.prod(1.0 - neg.to_numpy())),
                "event_count": int(len(group)),
                "direct_text_count": int(group["is_trump_direct_text"].sum()),
                "retweet_count": int(group["is_retweet"].sum()),
                "market_hours_event_count": int(group["in_market_hours"].sum()),
                "extreme_neg_count": int(group["extreme_neg_flag"].sum()),
                "extreme_pos_count": int(group["extreme_pos_flag"].sum()),
                "keyword_score_max": float(group["keyword_score_filled"].max()),
                "keyword_score_mean": float(group["keyword_score_filled"].mean()),
                "candidate_priority_high_count": int(group["candidate_priority_high_flag"].sum()),
                "source_twitter_count": int(group["source_twitter_flag"].sum()),
                "source_truthsocial_count": int(group["source_truthsocial_flag"].sum()),
                "term_first_count": int(group["term_first_flag"].sum()),
                "term_second_count": int(group["term_second_flag"].sum()),
                "min_event_time_utc": group["event_time_utc"].min(),
                "max_event_time_utc": group["event_time_utc"].max(),
            }
        )

    aggregated = pd.DataFrame(grouped_rows).sort_values("date").reset_index(drop=True)
    return aggregated


def add_temporal_text_features(daily):
    ordered = daily.sort_values("date").copy()
    signal_cols = [
        "finbert_neg_any",
        "finbert_neg_top1",
        "finbert_neg_top2_sum",
    ]
    for col in signal_cols:
        ordered[f"{col}_decay_2d"] = (
            ordered[col]
            + 0.5 * ordered[col].shift(1).fillna(0.0)
        )
        ordered[f"{col}_decay_3d"] = (
            ordered[col]
            + 0.5 * ordered[col].shift(1).fillna(0.0)
            + 0.25 * ordered[col].shift(2).fillna(0.0)
        )
    return ordered


aggregated_daily_raw = aggregate_daily_raw_rich_features(event_df_enriched)
aggregated_daily_raw = add_temporal_text_features(aggregated_daily_raw)

print("aggregated_daily_raw rows:", aggregated_daily_raw.shape)
display(aggregated_daily_raw.head())

aggregated_daily_raw rows: (741, 30)


,date,finbert_pos_max,finbert_pos_mean,finbert_neg_max,finbert_neg_mean,finbert_neu_mean,finbert_neg_top1,finbert_neg_top2_sum,finbert_neg_any,event_count,...,term_first_count,term_second_count,min_event_time_utc,max_event_time_utc,finbert_neg_any_decay_2d,finbert_neg_any_decay_3d,finbert_neg_top1_decay_2d,finbert_neg_top1_decay_3d,finbert_neg_top2_sum_decay_2d,finbert_neg_top2_sum_decay_3d
0,2017-01-26,0.048340,0.031359,0.958008,0.619853,0.348775,0.958008,1.814941,0.994260,3,...,3,0,2017-01-26 02:14:56+00:00,2017-01-26 13:55:03+00:00,0.994260,0.994260,0.958008,0.958008,1.814941,1.814941
1,2017-01-27,0.314453,0.189148,0.791992,0.402515,0.408325,0.791992,0.805031,0.794704,2,...,2,0,2017-01-27 13:19:10+00:00,2017-01-27 16:27:02+00:00,1.291835,1.291835,1.270996,1.270996,1.712502,1.712502
2,2017-02-01,0.051056,0.051056,0.027267,0.027267,0.921875,0.027267,0.027267,0.027267,1,...,1,0,2017-02-01 00:31:08+00:00,2017-02-01 00:31:08+00:00,0.424620,0.673185,0.423264,0.662766,0.429783,0.883518
3,2017-02-02,0.143555,0.134827,0.583984,0.304604,0.560547,0.583984,0.609207,0.594477,2,...,2,0,2017-02-02 11:34:36+00:00,2017-02-02 11:39:49+00:00,0.608111,0.806787,0.597618,0.795616,0.622841,0.824099
4,2017-02-06,0.040558,0.040558,0.068481,0.068481,0.891113,0.068481,0.068481,0.068481,1,...,1,0,2017-02-04 03:07:47+00:00,2017-02-04 03:07:47+00:00,0.365720,0.372537,0.360474,0.367290,0.373085,0.379902


## Merge With Daily Base

这一步固定使用 `date` 把 richer daily raw 表 merge 回统一的 daily base，
保持与当前 two-stage 相同的 split / target / controls。

In [8]:
def build_daily_raw_rich_table(daily_base, aggregated_daily_raw):
    merged = daily_base.merge(
        aggregated_daily_raw,
        on="date",
        how="left",
        validate="one_to_one",
    )

    fill_zero_cols = [
        "finbert_pos_max",
        "finbert_pos_mean",
        "finbert_neg_max",
        "finbert_neg_mean",
        "finbert_neu_mean",
        "finbert_neg_top1",
        "finbert_neg_top2_sum",
        "finbert_neg_any",
        "event_count",
        "direct_text_count",
        "retweet_count",
        "market_hours_event_count",
        "extreme_neg_count",
        "extreme_pos_count",
        "keyword_score_max",
        "keyword_score_mean",
        "candidate_priority_high_count",
        "source_twitter_count",
        "source_truthsocial_count",
        "term_first_count",
        "term_second_count",
        "finbert_neg_any_decay_2d",
        "finbert_neg_any_decay_3d",
        "finbert_neg_top1_decay_2d",
        "finbert_neg_top1_decay_3d",
        "finbert_neg_top2_sum_decay_2d",
        "finbert_neg_top2_sum_decay_3d",
    ]
    for col in fill_zero_cols:
        merged[col] = merged[col].fillna(0.0)

    int_cols = [
        "event_count",
        "direct_text_count",
        "retweet_count",
        "market_hours_event_count",
        "extreme_neg_count",
        "extreme_pos_count",
        "candidate_priority_high_count",
        "source_twitter_count",
        "source_truthsocial_count",
        "term_first_count",
        "term_second_count",
    ]
    for col in int_cols:
        merged[col] = merged[col].astype(int)

    merged["has_one_stage_signal"] = (merged["event_count"] > 0).astype(int)
    return merged


daily_raw_rich = build_daily_raw_rich_table(daily_base, aggregated_daily_raw)
print("daily_raw_rich rows:", daily_raw_rich.shape)
display(daily_raw_rich.head())

daily_raw_rich rows: (2293, 48)


,date,stage2_split,is_main_stage2_sample,gold_close,gold_ret_1d,dxy_ret_1d_lag1,real_yield_change_5d_lag1,vix_ma_5d_lag1,vix_change_5d_lag1,spx_ret_1d_lag1,...,term_second_count,min_event_time_utc,max_event_time_utc,finbert_neg_any_decay_2d,finbert_neg_any_decay_3d,finbert_neg_top1_decay_2d,finbert_neg_top1_decay_3d,finbert_neg_top2_sum_decay_2d,finbert_neg_top2_sum_decay_3d,has_one_stage_signal
0,2016-11-01,outside_main_sample,0,1302.519715,NaN,NaN,NaN,NaN,NaN,NaN,...,0,NaT,NaT,0.0,0.0,0.0,0.0,0.0,0.0,0
1,2016-11-02,outside_main_sample,0,1308.589123,0.004649,NaN,NaN,NaN,NaN,NaN,...,0,NaT,NaT,0.0,0.0,0.0,0.0,0.0,0.0,0
2,2016-11-03,outside_main_sample,0,1311.193116,0.001988,-0.000622,NaN,NaN,NaN,-0.006547,...,0,NaT,NaT,0.0,0.0,0.0,0.0,0.0,0.0,0
3,2016-11-04,outside_main_sample,0,1311.596841,0.000308,-0.001354,NaN,19.986667,NaN,-0.004433,...,0,NaT,NaT,0.0,0.0,0.0,0.0,0.0,0.0,0
4,2016-11-07,outside_main_sample,0,1299.637957,-0.009160,-0.001615,NaN,20.617500,NaN,-0.001668,...,0,NaT,NaT,0.0,0.0,0.0,0.0,0.0,0.0,0


## Quality Checks

这里必须明确验证：

- richer 表与 protocol split 是否一致
- richer 表是否只新增 raw groups，而没有混入 `P_taco` 或 label-window 派生列
- richer raw feature 缺失情况
- 多事件聚合分布

In [9]:
forbidden_present = [col for col in FORBIDDEN_FEATURE_COLS if col in daily_raw_rich.columns]
if forbidden_present:
    raise ValueError(f"Forbidden columns present in richer daily table: {forbidden_present}")

event_per_target = (
    event_df_enriched.groupby("target_trade_date_naive")
    .size()
    .rename("n_events")
    .reset_index()
)

qc_rows = [
    {"metric": "event_rows", "value": float(len(event_df_enriched))},
    {"metric": "daily_base_rows", "value": float(len(daily_base))},
    {"metric": "daily_raw_rich_rows", "value": float(len(daily_raw_rich))},
    {"metric": "stage2_split_mismatch_count", "value": float(split_mismatch_count)},
    {"metric": "signal_days", "value": float(daily_raw_rich["has_one_stage_signal"].sum())},
    {"metric": "signal_day_share", "value": float(daily_raw_rich["has_one_stage_signal"].mean())},
    {"metric": "target_dates_with_multi_events", "value": float((event_per_target["n_events"] > 1).sum())},
    {"metric": "target_dates_with_multi_events_share", "value": float((event_per_target["n_events"] > 1).mean())},
    {"metric": "max_events_same_target_trade_date", "value": float(event_per_target["n_events"].max())},
    {"metric": "mean_events_per_target_trade_date", "value": float(event_per_target["n_events"].mean())},
]

missing_feature_cols = [
    "finbert_pos_max",
    "finbert_neg_any",
    "event_count",
    "keyword_score_max",
    "candidate_priority_high_count",
    "source_twitter_count",
    "term_first_count",
] + CORE_CONTROL_FEATURES + TREND_FEATURES

for col in missing_feature_cols:
    qc_rows.append(
        {
            "metric": f"missing_rate::{col}",
            "value": float(daily_raw_rich[col].isna().mean()),
        }
    )

qc_summary = pd.DataFrame(qc_rows)

display(
    daily_raw_rich.groupby("stage2_split")
    .agg(
        n_days=("date", "count"),
        n_signal_days=("has_one_stage_signal", "sum"),
        avg_event_count=("event_count", "mean"),
        avg_keyword_score_max=("keyword_score_max", "mean"),
        avg_finbert_neg_any=("finbert_neg_any", "mean"),
    )
)
display(
    event_per_target["n_events"]
    .describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99])
    .to_frame("value")
)
display(qc_summary.head(20))

,n_days,n_signal_days,avg_event_count,avg_keyword_score_max,avg_finbert_neg_any
stage2_split,,,,,
outside_main_sample,1089,0,0.000000,0.000000,0.000000
primary_holdout,74,53,1.689189,4.472973,0.265999
train_pool,1130,687,1.557522,3.076991,0.248669


,value
count,741.000000
mean,2.545209
std,2.285009
min,1.000000
50%,2.000000
75%,3.000000
90%,5.000000
95%,7.000000
99%,11.600000
max,23.000000


,metric,value
0,event_rows,1886.000000
1,daily_base_rows,2293.000000
2,daily_raw_rich_rows,2293.000000
3,stage2_split_mismatch_count,0.000000
4,signal_days,740.000000
5,signal_day_share,0.322721
6,target_dates_with_multi_events,454.000000
7,target_dates_with_multi_events_share,0.612686
8,max_events_same_target_trade_date,23.000000
9,mean_events_per_target_trade_date,2.545209


In [10]:
plot_df = daily_raw_rich[daily_raw_rich["is_main_stage2_sample"] == 1].copy()

fig, axes = plt.subplots(4, 1, figsize=(16, 16), sharex=True)

sns.lineplot(data=plot_df, x="date", y="finbert_neg_any", ax=axes[0], linewidth=1.8)
axes[0].set_title("Rich single-stage daily finbert_neg_any")
axes[0].set_xlabel("")
axes[0].set_ylabel("neg_any")

sns.lineplot(data=plot_df, x="date", y="event_count", ax=axes[1], linewidth=1.6)
axes[1].set_title("Rich single-stage daily event_count")
axes[1].set_xlabel("")
axes[1].set_ylabel("event_count")

sns.lineplot(data=plot_df, x="date", y="keyword_score_max", ax=axes[2], linewidth=1.6)
axes[2].set_title("Rich single-stage daily keyword_score_max")
axes[2].set_xlabel("")
axes[2].set_ylabel("keyword_score_max")

sns.lineplot(data=plot_df, x="date", y="candidate_priority_high_count", ax=axes[3], linewidth=1.4)
axes[3].set_title("Rich single-stage daily candidate_priority_high_count")
axes[3].set_xlabel("date")
axes[3].set_ylabel("high_priority_count")

plt.tight_layout()
plt.show()

## Export

最终导出：

- event-to-day bridge
- richer daily raw feature 表
- QC summary

In [11]:
bridge_cols = [
    "event_id",
    "text_id",
    "source",
    "term_id",
    "event_time_utc",
    "event_date_ny",
    "feature_anchor_date",
    "target_trade_date",
    "target_trade_date_naive",
    "finbert_pos",
    "finbert_neu",
    "finbert_neg",
    "keyword_score_filled",
    "candidate_priority_high_flag",
    "is_trump_direct_text",
    "is_retweet",
    "in_market_hours",
    "source_twitter_flag",
    "source_truthsocial_flag",
    "term_first_flag",
    "term_second_flag",
]
bridge_export = event_df_enriched[bridge_cols].copy()

keep_daily_cols = [
    "date",
    "stage2_split",
    "is_main_stage2_sample",
    "gold_close",
    "gold_ret_1d",
    "finbert_pos_max",
    "finbert_pos_mean",
    "finbert_neg_max",
    "finbert_neg_mean",
    "finbert_neu_mean",
    "finbert_neg_top1",
    "finbert_neg_top2_sum",
    "finbert_neg_any",
    "finbert_neg_any_decay_2d",
    "finbert_neg_any_decay_3d",
    "finbert_neg_top1_decay_2d",
    "finbert_neg_top1_decay_3d",
    "finbert_neg_top2_sum_decay_2d",
    "finbert_neg_top2_sum_decay_3d",
    "event_count",
    "direct_text_count",
    "retweet_count",
    "market_hours_event_count",
    "extreme_neg_count",
    "extreme_pos_count",
    "has_one_stage_signal",
    "keyword_score_max",
    "keyword_score_mean",
    "candidate_priority_high_count",
    "source_twitter_count",
    "source_truthsocial_count",
    "term_first_count",
    "term_second_count",
] + CORE_CONTROL_FEATURES + TREND_FEATURES + [
    "min_event_time_utc",
    "max_event_time_utc",
]
daily_export = daily_raw_rich[keep_daily_cols].copy()

forbidden_in_export = [col for col in FORBIDDEN_FEATURE_COLS if col in daily_export.columns]
if forbidden_in_export:
    raise ValueError(f"Forbidden columns present in export: {forbidden_in_export}")

bridge_export.to_csv(BRIDGE_OUTPUT_PATH, index=False)
daily_export.to_csv(DAILY_OUTPUT_PATH, index=False)
qc_summary.to_csv(QC_OUTPUT_PATH, index=False)

print("Saved:")
print(" -", BRIDGE_OUTPUT_PATH)
print(" -", DAILY_OUTPUT_PATH)
print(" -", QC_OUTPUT_PATH)
print()
print("Daily richer feature columns:")
print(daily_export.columns.tolist())

Saved:
 - /Users/horace/Coding/ML finance/project/main_rerun/artifacts/ablation/one_stage_daily_raw_rich/one_stage_daily_raw_rich_event_daily_bridge_v1.csv
 - /Users/horace/Coding/ML finance/project/main_rerun/artifacts/ablation/one_stage_daily_raw_rich/one_stage_daily_raw_rich_features_v1.csv
 - /Users/horace/Coding/ML finance/project/main_rerun/artifacts/ablation/one_stage_daily_raw_rich/one_stage_daily_raw_rich_feature_qc_summary_v1.csv

Daily richer feature columns:
['date', 'stage2_split', 'is_main_stage2_sample', 'gold_close', 'gold_ret_1d', 'finbert_pos_max', 'finbert_pos_mean', 'finbert_neg_max', 'finbert_neg_mean', 'finbert_neu_mean', 'finbert_neg_top1', 'finbert_neg_top2_sum', 'finbert_neg_any', 'finbert_neg_any_decay_2d', 'finbert_neg_any_decay_3d', 'finbert_neg_top1_decay_2d', 'finbert_neg_top1_decay_3d', 'finbert_neg_top2_sum_decay_2d', 'finbert_neg_top2_sum_decay_3d', 'event_count', 'direct_text_count', 'retweet_count', 'market_hours_event_count', 'extreme_neg_count', 'ex

## Next Step

这份 notebook 完成后，下一步应新增一个 richer modeling spec：

- `single_stage_daily_raw_rich`

然后在统一 outer split 下，与当前两个历史 spec 一起比较：

- `macro_only`
- `one_stage_core`
- `single_stage_daily_raw_rich`